# Streaming

In [ ]:
import os
import pyrealsense2 as rs
import numpy as np
import cv2

In [ ]:
# config classes
pipeline = rs.pipeline()
config = rs.config()

# Get device product line for setting a supporting resolution
pipeline_wrapper = rs.pipeline_wrapper(pipeline)
pipeline_profile = config.resolve(pipeline_wrapper)

device = pipeline_profile.get_device()
device_product_line = str(device.get_info(rs.camera_info.product_line))

found_rgb = False
for s in device.sensors:
    if s.get_info(rs.camera_info.name) == "RGB Camera":
        found_rgb = True
        break
if not found_rgb:
    print("The demo requires Depth camera with Color sensor")
    exit(0)


In [ ]:
# depth
config.enable_stream(
    stream_type=rs.stream.depth, width=640, height=480, format=rs.format.z16, framerate=60
)
# color
config.enable_stream(
    stream_type=rs.stream.color, width=640, height=480, format=rs.format.bgr8, framerate=60
)
# infra
# config.enable_stream(
#     stream_type=rs.stream.infrared, width=640, height=480, format=rs.format.bgr8 , framerate=60
# )

# is selected config (for specific pipeline) valid
if config.can_resolve(pipeline):
    print('config OK')
else:
    print('baj van')

In [ ]:
# Start streaming
pipeline.start(config)

try:
    while True:

        # Wait for a coherent pair of frames: depth and color
        frames = pipeline.wait_for_frames()
        depth_frame = frames.get_depth_frame()
        color_frame = frames.get_color_frame()
        # infra_frame = frames.get_infrared_frame()

        if not depth_frame or not color_frame:
            continue

        # Convert images to numpy arrays
        depth_image = np.asanyarray(depth_frame.get_data())
        color_image = np.asanyarray(color_frame.get_data())
        # infra_image = np.asanyarray(infra_frame.get_data())

        # Apply colormap on depth image (image must be converted to 8-bit per pixel first)
        depth_colormap = cv2.applyColorMap(
            cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET
        )

        depth_colormap_dim = depth_colormap.shape
        color_colormap_dim = color_image.shape

        # If depth and color resolutions are different, resize color image to match depth image for display
        if depth_colormap_dim != color_colormap_dim:
            resized_color_image = cv2.resize(
                color_image,
                dsize=(depth_colormap_dim[1], depth_colormap_dim[0]),
                interpolation=cv2.INTER_AREA,
            )
            images = np.hstack((resized_color_image, depth_colormap))
        else:
            images = np.hstack((color_image, depth_colormap))
            # images = np.hstack((color_image, depth_colormap, infra_image))

        # Show images
        cv2.namedWindow("RealSense", cv2.WINDOW_AUTOSIZE)
        cv2.imshow("RealSense", images)

        # press q to exit without crash
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cv2.destroyAllWindows()

finally:

    # Stop streaming
    pipeline.stop()


In [ ]:
infra_image.shape, color_image.shape, depth_image.shape

In [ ]:
ord('e')

# Recording

In [ ]:
import os
import time
import pyrealsense2 as rs
import numpy as np
import cv2
import logging

## .db3

In [ ]:
# config classes
pipeline = rs.pipeline()
config = rs.config()
file_name = 'queue.db3'

# # # Get device product line for setting a supporting resolution
# pipeline_wrapper = rs.pipeline_wrapper(pipeline)
# pipeline_profile = config.resolve(pipeline_wrapper)

# device = pipeline_profile.get_device()
# device_product_line = str(device.get_info(rs.camera_info.product_line))

# depth
config.enable_stream(
    stream_type=rs.stream.depth, width=848, height=480, format=rs.format.z16, framerate=60
)
# color
config.enable_stream(
    stream_type=rs.stream.color, width=848, height=480, format=rs.format.bgr8, framerate=60
)
# enable recording
config.enable_record_to_file(file_name=file_name)
queue = rs.frame_queue(capacity=200, keep_frames=True)

# is selected config (for specific pipeline) valid
if config.can_resolve(pipeline):
    print('config OK')
else:
    print('baj van')

In [ ]:
import time

# Start streaming
pipeline.start(config, queue)

start = time.time()
current = 0

try:
    # while True:
    while current-start < 65:

        # Wait for a coherent pair of frames: depth and color
        frames = queue.wait_for_frame()
        frames = frames.as_frameset()
   
        depth_frame = frames.get_depth_frame()
        color_frame = frames.get_color_frame()
        # infra_frame = frames.get_infrared_frame()

        if not depth_frame or not color_frame:
            continue

        # Convert images to numpy arrays
        depth_image = np.asanyarray(depth_frame.get_data())
        color_image = np.asanyarray(color_frame.get_data())
        # infra_image = np.asanyarray(infra_frame.get_data())

        # Apply colormap on depth image (image must be converted to 8-bit per pixel first)
        depth_colormap = cv2.applyColorMap(
            cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET
        )

        depth_colormap_dim = depth_colormap.shape
        color_colormap_dim = color_image.shape

        # If depth and color resolutions are different, resize color image to match depth image for display
        if depth_colormap_dim != color_colormap_dim:
            resized_color_image = cv2.resize(
                color_image,
                dsize=(depth_colormap_dim[1], depth_colormap_dim[0]),
                interpolation=cv2.INTER_AREA,
            )
            images = np.hstack((resized_color_image, depth_colormap))
        else:
            images = np.hstack((color_image, depth_colormap))
            # images = np.hstack((color_image, depth_colormap, infra_image))


        current=time.time()

        # Show images
        cv2.namedWindow("RealSense", cv2.WINDOW_AUTOSIZE)
        cv2.imshow("RealSense", images)

        # press q to exit without crash
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cv2.destroyAllWindows()

finally:

    # Stop streaming
    pipeline.stop()


In [ ]:
help(rs.pipeline)

## .npy / .npz

In [ ]:
## License: Apache 2.0. See LICENSE file in root directory.
## Copyright(c) 2025 RealSense, Inc. All Rights Reserved.
#####################################################################################################
#                                                                                                  ##
#    Align depth to color with precaptured images in software device                               ##
#                                                                                                  ##
##  Purpose                                                                                        ##
##    This example first captures depth and color images from realsense camera and then            ##
##    demonstrate align depth to color with the precaptured images in software device              ##
##                                                                                                 ##
##  Steps:                                                                                         ##
##    1) stream realsense camera with depth 640x480@30fps and color 1280x720@30fps                 ##
##    2) capture camera depth and color intrinsics and extrinsics                                  ##
##    3) capture depth and color images and save into files in npy format                          ##
##    4) construct software device from the saved intrinsics, extrinsics, depth and color images   ##
##    5) align the precaptured depth image to color image                                         ##
##                                                                                                 ##
#####################################################################################################



fps = 30                  # frame rate
tv = 1000.0 / fps         # time interval between frames in miliseconds

max_num_frames  = 100      # max number of framesets to be captured into npy files and processed with software device

depth_file_name = "depth"  # depth_file_name + str(i) + ".npy"
color_file_name = "color"  # color_file_name + str(i) + ".npy"

# initialize intrinsics and extrinsics from the camera
camera_depth_intrinsics          = rs.intrinsics()  # camera depth intrinsics
camera_color_intrinsics          = rs.intrinsics()  # camera color intrinsics
camera_depth_to_color_extrinsics = rs.extrinsics()  # camera depth to color extrinsics


######################## start of first part - capture images from live device #######################################
# stream depth and color on attached realsnese camera and save depth and color frames into files with npy format
try:
    # create a context object, this object owns the handles to all connected realsense devices
    ctx = rs.context()
    devices = list(ctx.query_devices())
    
    if len(devices) > 0:
        print("Devices: {}".format(devices))
    else:
        print("No camera detected. Please connect a realsense camera and try again.")
        exit(0)
    
    pipeline = rs.pipeline()

    # configure streams
    config = rs.config()
    config.enable_stream(rs.stream.depth)
    config.enable_stream(rs.stream.color)

    # start streaming
    cfg = pipeline.start(config)
    
    # get scale
    depth_sensor = cfg.get_device().first_depth_sensor()
    depth_scale = depth_sensor.get_depth_scale()

    # get intrinsics
    camera_depth_profile = cfg.get_stream(rs.stream.depth)                                      # fetch depth depth stream profile
    camera_depth_intrinsics = camera_depth_profile.as_video_stream_profile().get_intrinsics()   # downcast to video_stream_profile and fetch intrinsics
    
    camera_color_profile = cfg.get_stream(rs.stream.color)                                      # fetch color stream profile
    camera_color_intrinsics = camera_color_profile.as_video_stream_profile().get_intrinsics()   # downcast to video_stream_profile and fetch intrinsics
    
    camera_depth_to_color_extrinsics = camera_depth_profile.get_extrinsics_to(camera_color_profile)
 

    print("camera depth intrinsic:", camera_depth_intrinsics)
    print("camera color intrinsic:", camera_color_intrinsics)
    print("camera depth to color extrinsic:", camera_depth_to_color_extrinsics)

    print("streaming attached camera and save depth and color frames into files in npy format ...")

    i = 0
    while i < max_num_frames:
        # wait until a new coherent set of frames is available on the device
        frames = pipeline.wait_for_frames()
        depth = frames.get_depth_frame()
        color = frames.get_color_frame()

        # Validate that both frames are valid
        if not depth or not color:
            continue
        
        # convert images to numpy arrays
        depth_image = np.asanyarray(depth.get_data())
        color_image = np.asanyarray(color.get_data())
        # save images in npy format
        depth_file = depth_file_name + str(i) + ".npy"
        color_file = color_file_name + str(i) + ".npy"
        print("saving frame set ", i, depth_file, color_file)
        
        with open(depth_file, 'wb') as f1:
            np.save(f1,depth_image)
        
        with open(color_file, 'wb') as f2:
            np.save(f2,color_image)

        # next frameset
        i = i +1

except Exception as e:
    logging.error("An error occurred: %s", e, exc_info=True)
    exit(1)


In [ ]:

######################## end of first part - capture images from live device #######################################



######################## start of second part - align depth to color in software device #############################
# align depth to color with the above precaptured images in software device

# software device
sdev = rs.software_device()

# software depth sensor
depth_sensor: rs.software_sensor = sdev.add_sensor("Depth")

# depth instrincis
depth_intrinsics = rs.intrinsics()

depth_intrinsics.width  = camera_depth_intrinsics.width
depth_intrinsics.height = camera_depth_intrinsics.height

depth_intrinsics.ppx = camera_depth_intrinsics.ppx
depth_intrinsics.ppy = camera_depth_intrinsics.ppy

depth_intrinsics.fx = camera_depth_intrinsics.fx
depth_intrinsics.fy = camera_depth_intrinsics.fy

depth_intrinsics.coeffs = camera_depth_intrinsics.coeffs       ## [0.0, 0.0, 0.0, 0.0, 0.0]
depth_intrinsics.model = camera_depth_intrinsics.model         ## rs.pyrealsense2.distortion.brown_conrady

#depth stream
depth_stream = rs.video_stream()
depth_stream.type = rs.stream.depth
depth_stream.width = depth_intrinsics.width
depth_stream.height = depth_intrinsics.height
depth_stream.fps = fps
depth_stream.bpp = 2                              # depth z16 2 bytes per pixel
depth_stream.fmt = rs.format.z16
depth_stream.intrinsics = depth_intrinsics
depth_stream.index = 0
depth_stream.uid = 1

depth_profile = depth_sensor.add_video_stream(depth_stream)

# software color sensor
color_sensor: rs.software_sensor = sdev.add_sensor("Color")

# color intrinsic:
color_intrinsics = rs.intrinsics()
color_intrinsics.width = camera_color_intrinsics.width
color_intrinsics.height = camera_color_intrinsics.height

color_intrinsics.ppx = camera_color_intrinsics.ppx
color_intrinsics.ppy = camera_color_intrinsics.ppy

color_intrinsics.fx = camera_color_intrinsics.fx
color_intrinsics.fy = camera_color_intrinsics.fy

color_intrinsics.coeffs = camera_color_intrinsics.coeffs
color_intrinsics.model = camera_color_intrinsics.model

color_stream = rs.video_stream()
color_stream.type = rs.stream.color
color_stream.width = color_intrinsics.width
color_stream.height = color_intrinsics.height
color_stream.fps = fps
color_stream.bpp = 3                                # color stream rgb8 3 bytes per pixel in this example
color_stream.fmt = rs.format.rgb8
color_stream.intrinsics = color_intrinsics
color_stream.index = 0
color_stream.uid = 2

color_profile = color_sensor.add_video_stream(color_stream)

# depth to color extrinsics     auto z_to_other = depth_profile.get_extrinsics_to(other_profile);
depth_to_color_extrinsics = rs.extrinsics()
depth_to_color_extrinsics.rotation = camera_depth_to_color_extrinsics.rotation
depth_to_color_extrinsics.translation = camera_depth_to_color_extrinsics.translation
depth_profile.register_extrinsics_to(color_profile, depth_to_color_extrinsics)

# start software sensors
depth_sensor.open(depth_profile)
color_sensor.open(color_profile)

# syncronize frames from depth and color streams
camera_syncer = rs.syncer()
depth_sensor.start(camera_syncer)
color_sensor.start(camera_syncer)

# create a depth alignment object
# rs.align allows us to perform alignment of depth frames to others frames
# the "align_to" is the stream type to which we plan to align depth frames
# align depth frame to color frame
align_to = rs.stream.color
align = rs.align(align_to)

# colorizer for depth rendering
colorizer = rs.colorizer()

paused = False

# loop through pre-captured frames
for i in range(0, max_num_frames):
    print("\nframe set:", i)
    
    # precaptured depth and color image files in npy format
    df = depth_file_name + str(i) + ".npy"
    cf = color_file_name + str(i) + ".npy"

    if (not os.path.exists(cf)) or (not os.path.exists(df)): continue

    # load depth frame from precaptured npy file
    print('loading depth frame ', df)
    depth_npy = np.load(df, mmap_mode='r')

    # create software depth frame
    depth_swframe = rs.software_video_frame()
    depth_swframe.stride = depth_stream.width * depth_stream.bpp
    depth_swframe.bpp = depth_stream.bpp
    depth_swframe.timestamp = i * tv
    depth_swframe.pixels = depth_npy
    depth_swframe.domain = rs.timestamp_domain.hardware_clock
    depth_swframe.frame_number = i
    depth_swframe.profile = depth_profile.as_video_stream_profile()
    depth_swframe.depth_units = depth_scale
    depth_sensor.on_video_frame(depth_swframe)

    # load color frame from precaptured npy file
    print('loading color frame ', cf)
    color_npy = np.load(cf, mmap_mode='r')
 
    # create software color frame
    color_swframe = rs.software_video_frame()
    color_swframe.stride = color_stream.width * color_stream.bpp
    color_swframe.bpp = color_stream.bpp
    color_swframe.timestamp = i * tv
    color_swframe.pixels = color_npy
    color_swframe.domain = rs.timestamp_domain.hardware_clock
    color_swframe.frame_number = i
    color_swframe.profile = color_profile.as_video_stream_profile()
    color_sensor.on_video_frame(color_swframe)
    
    # synchronize depth and color, receive as frameset
    frames = camera_syncer.wait_for_frames()
    print("frame set:", frames.size(), " ", frames)

    # get unaligned depth frame
    unaligned_depth_frame = frames.get_depth_frame()
    if not unaligned_depth_frame: continue

    # align depth frame to color 
    aligned_frames = align.process(frames)

    aligned_depth_frame = aligned_frames.get_depth_frame()
    color_frame = aligned_frames.get_color_frame()

    if (not aligned_depth_frame) or (not color_frame): continue

    aligned_depth_frame = colorizer.colorize(aligned_depth_frame)    
    npy_aligned_depth_image = np.asanyarray(aligned_depth_frame.get_data())

    npy_color_image = np.asanyarray(color_frame.get_data())

    # render aligned images:
    # depth align to color
    # aligned depth on left
    # color on right    
    images = np.hstack((npy_aligned_depth_image, npy_color_image))
    cv2.namedWindow('Align Example', cv2.WINDOW_NORMAL)
    cv2.imshow('Align Example', images)
    key = cv2.waitKey(1)

    # render original unaligned depth as reference
    # colorized_unaligned_depth_frame = colorizer.colorize(unaligned_depth_frame)
    # npy_unaligned_depth_image = np.asanyarray(colorized_unaligned_depth_frame.get_data())
    # cv2.imshow("Unaligned Depth", npy_unaligned_depth_image)
    
    # press ENTER or SPACEBAR key to pause the image window for 5 seconds

    if key == 13 or key == 32: paused = not paused
        
    if paused:
        print("Paused for 5 seconds ...", i, ", press ENTER or SPACEBAR key anytime for additional pauses.")
        time.sleep(5)
        paused = not paused

# end of second part - align depth to color with the precaptured images in software device
######################## End of second part - align depth to color in software device #############################
    
cv2.destroyAllWindows()

# Playback from file

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2
import argparse
import os.path

In [ ]:
# context = rs.context()
# playback_device = context.load_device(filename='./faszom.db3')
# pipeline.set_device(playback_device)

file_name = 'queue.db3'

pipeline = rs.pipeline()
config = rs.config()

config.enable_device_from_file(file_name=file_name, repeat_playback=False)

config.enable_stream(
    stream_type=rs.stream.depth, width=848, height=480, format=rs.format.z16, framerate=60
)
config.enable_stream(
    stream_type=rs.stream.color, width=848, height=480, format=rs.format.bgr8, framerate=60
)
if config.can_resolve(pipeline):
    print('config OK')

In [ ]:
# Start streaming from file
pipeline.start(config)

# Create opencv window to render image in
cv2.namedWindow("Depth Stream", cv2.WINDOW_AUTOSIZE)

# Create colorizer object
colorizer = rs.colorizer()
frames = None
# Streaming loop
# while True:
success = True
frame_meta = []
while success:

    success, frames = pipeline.try_wait_for_frames(timeout_ms=500)
    # frames = pipeline.poll_for_frames()

    if frames:
        # frame_meta.append([frames.get_frame_number(), frames.get_timestamp()])
        # print([frames.get_frame_number(), frames.get_timestamp()])
        # frame_meta.append([frames.get_color_frame().frame_number,frames.get_color_frame().timestamp, frames.get_depth_frame().frame_number, frames.get_depth_frame().timestamp])
        # Get depth frame
        depth_frame = frames.get_depth_frame()
        # Colorize depth frame to jet colormap
        depth_color_frame = colorizer.colorize(depth_frame)
        # Convert depth_frame to numpy array to render image in opencv
        depth_color_image = np.asanyarray(depth_color_frame.get_data())

        color_frame = frames.get_color_frame()
        color_image = np.asanyarray(color_frame.get_data())

        images = np.hstack((color_image, depth_color_image))
    
        cv2.imshow('Playback', images)
    key = cv2.waitKey(1)
    # if pressed escape exit program
    if key == 27 or key==ord('q'):
        cv2.destroyAllWindows()
        break
cv2.destroyAllWindows()
pipeline.stop()

In [ ]:
help(rs.pipeline.try_wait_for_frames)

# Align frames, save color video, save depth data

In [1]:
import os
import time
import pyrealsense2 as rs
import numpy as np
import cv2

input_file_name = "t1.db3"
out_file_name_color = os.path.splitext(input_file_name)[0] + "_color.avi"
out_file_name_infra = os.path.splitext(input_file_name)[0] + "_infra.avi"

# config classes
pipeline = rs.pipeline()
config = rs.config()
config.enable_device_from_file(file_name=input_file_name, repeat_playback=False)
queue = rs.frame_queue(capacity=2000, keep_frames=True)

recording_framerate = 60
width = 848
height = 480

# depth
config.enable_stream(
    stream_type=rs.stream.depth,
    width=width,
    height=height,
    format=rs.format.z16,
    framerate=recording_framerate,
)
# color
config.enable_stream(
    stream_type=rs.stream.color,
    width=width,
    height=height,
    format=rs.format.bgr8,
    framerate=recording_framerate,
)
# infra (only enable if it was recorded)
# config.enable_stream(
#     stream_type=rs.stream.infrared,
#     width=width,
#     height=height,
#     format=rs.format.bgr8,
#     framerate=recording_framerate,
# )

# is selected config (for specific pipeline) valid
if config.can_resolve(pipeline):
    print("config OK")
else:
    print("baj van")

# opencv video writer
fourcc = cv2.VideoWriter_fourcc(*"XVID")
video_writer_color = cv2.VideoWriter(
    filename=out_file_name_color, fourcc=fourcc, fps=recording_framerate, frameSize=(width, height)
)
# video_writer_infra = cv2.VideoWriter(
#     filename=out_file_name_infra, fourcc=fourcc, fps=recording_framerate, frameSize=(width, height)
# )


# alignment parameters
align_to = rs.stream.color
align = rs.align(align_to)

config OK


In [4]:
depth_frames = np.empty(shape=(1, height, width))
pipeline.start(config, queue)

frame_num = 0
while True:
    
    # Wait for a coherent pair of frames
    success, frames = queue.try_wait_for_frame(timeout_ms=1000)
    if not success:
        break

    frames = frames.as_frameset()
    aligned_frames = align.process(frames)

    depth_frame = aligned_frames.get_depth_frame()
    color_frame = aligned_frames.get_color_frame()
    infra_frame = aligned_frames.get_infrared_frame()

    if not depth_frame or not color_frame:
        continue

    # Convert images to numpy arrays
    depth_image = np.asanyarray(depth_frame.get_data())
    color_image = np.asanyarray(color_frame.get_data())
    # infra_image = np.asanyarray(infra_frame.get_data())

    depth_frames = np.vstack((depth_frames, np.expand_dims(depth_image, axis=0)))

    # write color image (to video)
    video_writer_color.write(color_image)
    # video_writer_infra.write(infra_image)

    frame_num += 1
    print(
        frame_num,
        color_frame.get_frame_number(),
        # infra_frame.get_frame_number(),
        depth_frame.get_frame_number(),
    )

# stop video writer
video_writer_color.release()
# video_writer_infra.release()

pipeline.stop()

1 1 5
2 2 6
3 3 7
4 4 8
5 4 9
6 5 9
7 6 10
8 7 11
9 8 12
10 8 13
11 9 13
12 10 14
13 11 15
14 12 16
15 13 16
16 14 16
17 15 16
18 16 16
19 16 54
20 16 55
21 16 56
22 16 57
23 16 58
24 55 58
25 55 60
26 57 61
27 58 61
28 59 63
29 60 64
30 61 65
31 62 66
32 63 67
33 64 68
34 65 69
35 66 70
36 67 71
37 68 72
38 69 73
39 70 74
40 71 75
41 72 76
42 73 77
43 74 78
44 75 79
45 76 80
46 77 81
47 78 82
48 79 83
49 80 84
50 81 85
51 82 86
52 83 87
53 84 88
54 85 89
55 86 90
56 87 91
57 88 92
58 89 93
59 89 94
60 90 94
61 91 95
62 92 96
63 93 97
64 94 98
65 95 99
66 96 100
67 97 101
68 98 102
69 99 103
70 100 104
71 101 105
72 102 106
73 103 107
74 104 108
75 105 109
76 106 110
77 107 111
78 108 112
79 109 113
80 110 114
81 111 115
82 112 116
83 113 117
84 114 118
85 115 119
86 116 120
87 117 121
88 118 122
89 119 123
90 120 124
91 121 125
92 122 126
93 123 127
94 124 128
95 125 129
96 126 130
97 127 131
98 128 132
99 129 133
100 130 134
101 131 135
102 132 136
103 133 137
104 134 138
105 135 139

In [ ]:
depth_frames

In [5]:
np.save('t1_depth_frames', depth_frames)

In [ ]:
d = np.load('./depth_frames.npy')
(d == depth_frames).all()

# Pose estimation

In [ ]:
from rtmlib import Body, Custom, PoseTracker, draw_skeleton
import cv2
import importlib
import numpy as np
import pyrealsense2 as rs


rtmlib_module = importlib.import_module("rtmlib")


file_name = 't1.db3'

pipeline = rs.pipeline()
config = rs.config()
queue = rs.frame_queue(capacity=7200, keep_frames=True)


config.enable_device_from_file(file_name=file_name, repeat_playback=True)

config.enable_stream(
    stream_type=rs.stream.depth, width=848, height=480, format=rs.format.z16, framerate=60
)
config.enable_stream(
    stream_type=rs.stream.color, width=848, height=480, format=rs.format.bgr8, framerate=60
)
if config.can_resolve(pipeline):
    print('config OK')

In [ ]:
class Custom:

    def __init__(
        self,
        det_class: str = None,
        det: str = None,
        det_input_size: tuple = (640, 640),
        pose_class: str = None,
        pose: str = None,
        pose_input_size: tuple = (192, 256),
        mode: str = None,
        to_openpose: bool = False,
        backend: str = "onnxruntime",
        device: str = "cuda",
    ):

        if det_class is not None:
            try:
                det_class = getattr(rtmlib_module, det_class)
                self.det_model = det_class(
                    det, model_input_size=det_input_size, backend=backend, device=device
                )
                self.one_stage = False

            except ImportError:
                raise ImportError(f"{det_class} is not supported by rtmlib.")
        else:
            self.one_stage = True

        if pose_class is not None:
            try:
                pose_class = getattr(rtmlib_module, pose_class)
                self.pose_model = pose_class(
                    pose,
                    model_input_size=pose_input_size,
                    to_openpose=to_openpose,
                    backend=backend,
                    device=device,
                )
            except ImportError:
                raise ImportError(f"{pose_class} is not supported by rtmlib.")

    MODE = {
        "performance": {
            "det": "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/yolox_x_8xb8-300e_humanart-a39d44ed.zip",  # noqa
            "det_input_size": (640, 640),
            "pose": "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.zip",  # noqa
            "pose_input_size": (288, 384),
        },
        "lightweight": {
            "det": "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/yolox_tiny_8xb8-300e_humanart-6f3252f9.zip",  # noqa
            "det_input_size": (416, 416),
            "pose": "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-s_simcc-body7_pt-body7_420e-256x192-acd4a1ef_20230504.zip",  # noqa
            "pose_input_size": (192, 256),
        },
        "balanced": {
            "det": "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/yolox_m_8xb8-300e_humanart-c2c7a14a.zip",  # noqa
            "det_input_size": (640, 640),
            "pose": "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip",  # noqa
            "pose_input_size": (192, 256),
        },
    }

    RTMO_MODE = {
        "performance": {
            "pose": "https://download.openmmlab.com/mmpose/v1/projects/rtmo/onnx_sdk/rtmo-l_16xb16-600e_body7-640x640-b37118ce_20231211.zip",  # noqa
            "pose_input_size": (640, 640),
        },
        "lightweight": {
            "pose": "https://download.openmmlab.com/mmpose/v1/projects/rtmo/onnx_sdk/rtmo-s_8xb32-600e_body7-640x640-dac2bf74_20231211.zip",  # noqa
            "pose_input_size": (640, 640),
        },
        "balanced": {
            "pose": "https://download.openmmlab.com/mmpose/v1/projects/rtmo/onnx_sdk/rtmo-m_16xb16-600e_body7-640x640-39e78cc4_20231211.zip",  # noqa
            "pose_input_size": (640, 640),
        },
    }

    def __call__(self, image: np.ndarray):
        if self.one_stage:
            keypoints, scores = self.pose_model(image)
        else:
            bboxes = self.det_model(image)
            keypoints, scores = self.pose_model(image, bboxes=bboxes)

        return keypoints, scores


In [ ]:
device = 'cuda'
backend = 'onnxruntime'
openpose_skeleton = False
custom = Custom(
    det_class="YOLOX",  #'RTMDet',
    det="https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/yolox_nano_8xb8-300e_humanart-40f6f0d0.zip",
    det_input_size=(416,416),
    pose_class="RTMPose",
    pose='https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-x_simcc-body7_pt-body7-halpe26_700e-384x288-7fb6e239_20230606.zip',
    pose_input_size=(288,384),
    backend=backend,
    device=device,
)


# Create opencv window to render image in
cv2.namedWindow("Depth Stream", cv2.WINDOW_AUTOSIZE)

pipeline.start(config, queue)
# pipeline.start(config)

# Create colorizer object
colorizer = rs.colorizer()
frames = None
# Streaming loop
# while True:
while frames != rs.frame():

    # frames = pipeline.wait_for_frames()
    frames = queue.wait_for_frame()
    frames = frames.as_frameset()
    # frames = pipeline.poll_for_frames()

    if frames:
        print(frames)
        # Get depth frame
        depth_frame = frames.get_depth_frame()


        # Colorize depth frame to jet colormap
        depth_color_frame = colorizer.colorize(depth_frame)
        # Convert depth_frame to numpy array to render image in opencv
        depth_color_image = np.asanyarray(depth_color_frame.get_data())

        color_frame = frames.get_color_frame()
        color_image = np.asanyarray(color_frame.get_data())

        color_image = cv2.rotate(color_image, cv2.ROTATE_90_COUNTERCLOCKWISE)
        depth_color_image = cv2.rotate(depth_color_image, cv2.ROTATE_90_COUNTERCLOCKWISE)


        # pose estimation
        keypoints, scores = custom(color_image)
        if keypoints.shape != (1, 26, 2):
            keypoints = keypoints[0, :, :]
            scores = scores[0, :]

        for kpt in keypoints[:, :]:
            cv2.circle(color_image, (int(kpt[0]), int(kpt[1])), 3, (0, 255, 0), 2)
            cv2.circle(depth_color_image, (int(kpt[0]), int(kpt[1])), 3, (0, 255, 0), 2)



        images = np.hstack((color_image, depth_color_image))

        cv2.imshow('Playback', images)
    key = cv2.waitKey(1)
    # if pressed escape exit program
    if key == 27 or key==ord('q'):
        cv2.destroyAllWindows()
        break
cv2.destroyAllWindows()
pipeline.stop()

In [ ]:
frames.data

In [ ]:
help(rs.frame_queue)

In [ ]:
help(rs.pipeline.poll_for_frames)

# frame que test

In [ ]:
import pyrealsense2 as rs
import time

# Implement two "processing" functions, each of which
# occassionally lags and takes longer to process a frame.
def slow_processing(frame):
    n = frame.get_frame_number() 
    if n % 20 == 0:
        time.sleep(1/4)
    print(n)

def slower_processing(frame):
    n = frame.get_frame_number() 
    if n % 20 == 0:
        time.sleep(1)
    print(n)

def fast_processing(frame):
    n = frame.get_frame_number()
    print(n)

frame_numbers = {
    "fast": [],
    "slow": [],
    "slow_que": [],
    "slower": [],
    "slower_que": [],
}

try:

    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)



    # FAST
    print('Fast callback')
    pipeline.start(config)
    start = time.time()
    while time.time() - start < 5:
        frames = pipeline.wait_for_frames()
        frame_numbers['fast'].append(frames.get_frame_number())
        fast_processing(frames)
    pipeline.stop()


    # SLOW
    # Start streaming to the slow processing function.
    # This stream will lag, causing the occasional frame drop.
    print("Slow callback")
    pipeline.start(config)
    start = time.time()
    while time.time() - start < 5:
        frames = pipeline.wait_for_frames()
        frame_numbers['slow'].append(frames.get_frame_number())
        slow_processing(frames)
    pipeline.stop()

    # SLOW + QUE
    # Start streaming to the the slow processing function by way of a frame queue.
    # This stream will occasionally hiccup, but the frame_queue will prevent frame loss.
    print("Slow callback + queue")
    queue = rs.frame_queue(50)
    pipeline.start(config, queue)
    start = time.time()
    while time.time() - start < 5:
        frames = queue.wait_for_frame()
        frame_numbers['slow_que'].append(frames.get_frame_number())
        slow_processing(frames)
    pipeline.stop()

    # SLOWER
    # Start streaming to the the slower processing function by way of a frame queue.
    # This stream will drop frames because the delays are long enough that the backed up
    # frames use the entire internal frame pool preventing the SDK from creating more frames.
    print("Slower callback + queue")
    queue = rs.frame_queue(50)
    pipeline.start(config, queue)
    start = time.time()
    while time.time() - start < 5:
        frames = queue.wait_for_frame()
        frame_numbers['slower'].append(frames.get_frame_number())
        slower_processing(frames)
    pipeline.stop()

    # SLOWER + QUE
    # Start streaming to the slower processing function by way of a keeping frame queue.
    # This stream will no longer drop frames because the frame queue tells the SDK
    # to remove the frames it holds from the internal frame queue, allowing the SDK to
    # allocate space for and create more frames .
    print("Slower callback + keeping queue")
    queue = rs.frame_queue(capacity=200, keep_frames=True)
    pipeline.start(config, queue)
    start = time.time()
    while time.time() - start < 5:
        frames = queue.wait_for_frame()
        frame_numbers['slower_que'].append(frames.get_frame_number())
        slower_processing(frames)
    pipeline.stop()

except Exception as e:
    print(e)
except:
    print("A different Error")
else:
    print("Done")

for key, value in frame_numbers.items():
    print(key, len(value))

In [ ]:
# type(frames)
frames.as_frameset()

In [ ]:
help(rs.composite_frame)

In [ ]:
for ind, frame in enumerate(frame_numbers['slower_que'],1):
    print(ind, frame)

In [ ]:
help(rs.frame_queue)

# Alignment test

In [ ]:
import matplotlib.pyplot as plt 

colorizer = rs.colorizer()
color = np.asanyarray(color_frame.get_data())

align = rs.align(align_to=rs.stream.color) # align_to=rs.stream.color/infra... -> aligns depth to other, align_to=RS2_STREAM_DEPTH -> align other stream to depth 
frameset = align.process(frames)

aligned_depth_frame = frameset.get_depth_frame()
colorized_depth = np.asanyarray(colorizer.colorize(aligned_depth_frame).get_data())

images = np.hstack((color, colorized_depth))
plt.imshow(images)

## Align while streaming/recording

In [ ]:
import os
import time
import pyrealsense2 as rs
import numpy as np
import cv2
import logging
import time

# config classes
pipeline = rs.pipeline()
config = rs.config()
file_name = 'align.db3'
out_file_name_color = os.path.splitext(file_name)[0] + 'color.avi'
out_file_name_infra = os.path.splitext(file_name)[0] + 'infra.avi'

recording_framerate = 60
width = 848
height = 480

# depth
config.enable_stream(
    stream_type=rs.stream.depth, width=width, height=height, format=rs.format.z16, framerate=recording_framerate
)
# color
config.enable_stream(
    stream_type=rs.stream.color, width=width, height=height, format=rs.format.bgr8, framerate=recording_framerate
)
# infra
config.enable_stream(
    stream_type=rs.stream.infrared, width=width, height=height, format=rs.format.bgr8, framerate=recording_framerate
)


# enable recording
config.enable_record_to_file(file_name=file_name)
queue = rs.frame_queue(capacity=2000, keep_frames=True)

# is selected config (for specific pipeline) valid
if config.can_resolve(pipeline):
    print('config OK')
else:
    print('baj van')

# opencv video writer
fourcc = cv2.VideoWriter_fourcc(*"XVID")
video_writer_color = cv2.VideoWriter(filename=out_file_name_color, fourcc=fourcc, fps=recording_framerate, frameSize=(width,height))
video_writer_infra = cv2.VideoWriter(filename=out_file_name_infra, fourcc=fourcc, fps=recording_framerate, frameSize=(width,height))



align_to = rs.stream.color
align = rs.align(align_to)

In [ ]:
# Start streaming
pipeline.start(config, queue)

start = time.time()
current_time = 0

frame_num = 0

depth_frames = np.empty(shape=(1,480,848))

try:
    while current_time-start < 65:

        # Wait for a coherent pair of frames: depth and color
        frames = queue.wait_for_frame()
        frames = frames.as_frameset()
        aligned_frames = align.process(frames)
   
        depth_frame = aligned_frames.get_depth_frame()
        color_frame = aligned_frames.get_color_frame()
        infra_frame = aligned_frames.get_infrared_frame()

        if not depth_frame or not color_frame or not infra_frame:
            continue

        # Convert images to numpy arrays
        depth_image = np.asanyarray(depth_frame.get_data())
        color_image = np.asanyarray(color_frame.get_data())
        infra_image = np.asanyarray(infra_frame.get_data())

        depth_frames = np.vstack((depth_frames,np.expand_dims(depth_image, axis=0)))

        # write color image (to video)
        video_writer_color.write(color_image)
        video_writer_infra.write(infra_image)
        frame_num+=1
        print(frame_num, color_frame.get_frame_number(), infra_frame.get_frame_number(), depth_frame.get_frame_number())

        # # Apply colormap on depth image (image must be converted to 8-bit per pixel first)
        # depth_colormap = cv2.applyColorMap(
        #     cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET
        # )

        # depth_colormap_dim = depth_colormap.shape
        # color_colormap_dim = color_image.shape

        # # If depth and color resolutions are different, resize color image to match depth image for display
        # if depth_colormap_dim != color_colormap_dim:
        #     resized_color_image = cv2.resize(
        #         color_image,
        #         dsize=(depth_colormap_dim[1], depth_colormap_dim[0]),
        #         interpolation=cv2.INTER_AREA,
        #     )
        #     images = np.hstack((resized_color_image, depth_colormap))
        # else:
        #     images = np.hstack((color_image, depth_colormap))
        #     # images = np.hstack((color_image, depth_colormap, infra_image))


        current_time=time.time()

        # Show images
        # cv2.namedWindow("RealSense", cv2.WINDOW_AUTOSIZE)
        # cv2.imshow("RealSense", images)

        # press q to exit without crash
    #     if cv2.waitKey(1) & 0xFF == ord("q"):
    #         break
    # cv2.destroyAllWindows()

finally:
    # stop video writer
    video_writer_color.release()
    video_writer_infra.release()

    # Stop streaming
    pipeline.stop()


In [ ]:
depth_frames

In [ ]:
d1 = depth_image
d2 = np.copy(depth_image)

asdf = np.stack([d1,d2])
asdf.shape

In [ ]:
d1.shape, np.expand_dims(d1,axis=0).shape, asdf.shape

In [ ]:
np.vstack((asdf,np.expand_dims(d1, axis=0)))

# check .db3 file

In [ ]:
import sqlite3


dbfile = './t1.db3'
# Create a SQL connection to our SQLite database
con = sqlite3.connect(dbfile)

# creating cursor
cur = con.cursor()

# reading all table names
# table_list = [a for a in cur.execute("SELECT name FROM sqlite_master WHERE type = 'table'")]
table_list = [a for a in cur.execute("SELECT name FROM sqlite_master WHERE type = 'table'")]
# here is you table list
print(table_list)

# Be sure to close the connection
con.close()